# Taller: Pipelines de Sklearn

## Escenario:

Estás trabajando como Data Scientist en una consultora. Has entrenado un modelo que predice
si un pasajero del Titanic habría sobrevivido (sí, el dataset de siempre — pero el
problema que vamos a resolver hoy es 100% real y os lo vais a encontrar en vuestro
primer trabajo).

Mañana, el equipo de Ingeniería va a coger tu modelo y lo va a meter en un servicio
que recibe pasajeros nuevos cada minuto. **Ellos no van a leer tu notebook. No van a
copiar tus funciones de limpieza. Van a hacer, literalmente, dos líneas:**

```python
modelo = cargar_modelo("modelo.joblib")
modelo.predict(datos_nuevos)
```

Si tu modelo no puede sobrevivir a eso, no importa lo bueno que sea. Hoy vamos a ver
por qué `Pipeline` de sklearn es la herramienta que hace que esas dos líneas funcionen
— y qué pasa cuando no lo usas.


## Parte 0 — Setup

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pd.options.mode.copy_on_write = True

In [2]:
def data_report(df):
    '''Describe los campos de un dataframe: tipo, % missings, cardinalidad.'''
    cols = pd.DataFrame(df.columns.values, columns=["COL_N"])
    types = pd.DataFrame(df.dtypes.values, columns=["DATA_TYPE"])
    percent_missing = round(df.isnull().sum() * 100 / len(df), 2)
    percent_missing_df = pd.DataFrame(percent_missing.values, columns=["MISSINGS (%)"])
    unicos = pd.DataFrame(df.nunique().values, columns=["UNIQUE_VALUES"])
    percent_cardin = round(unicos["UNIQUE_VALUES"] * 100 / len(df), 2)
    percent_cardin_df = pd.DataFrame(percent_cardin.values, columns=["CARDIN (%)"])
    concatenado = pd.concat([cols, types, percent_missing_df, unicos, percent_cardin_df],
                            axis=1, sort=False)
    concatenado.set_index("COL_N", drop=True, inplace=True)
    return concatenado.T

In [3]:
train = pd.read_csv("./data/titanic_train.csv")
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
data_report(train)

COL_N,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
DATA_TYPE,int64,int64,int64,object,object,float64,int64,int64,object,float64,object,object
MISSINGS (%),0.0,0.0,0.0,0.0,0.0,19.87,0.0,0.0,0.0,0.0,77.1,0.22
UNIQUE_VALUES,891,2,3,891,2,88,7,7,681,248,147,3
CARDIN (%),100.0,0.22,0.34,100.0,0.22,9.88,0.79,0.79,76.43,27.83,16.5,0.34


## Parte 1 — Decisiones de limpieza (resumen)

No vamos a hacer el EDA completo en clase — eso ya lo sabéis hacer. Vamos directos
a las decisiones, que es lo que necesitamos para construir el pipeline:

| Columna | Decisión | Por qué |
|---|---|---|
| `PassengerId`, `Name` | Excluir | Identificadores únicos, sin señal predictiva |
| `Cabin` | Excluir | 77% de missings, y del 23% informado casi todos son valores únicos → no imputable |
| `Ticket` | Excluir | Alta cardinalidad, texto libre sin estructura aprovechable sin trabajo extra |
| `Age` | Imputar (mediana) | ~20% de missings, variable continua con outliers → mediana mejor que media |
| `Embarked` | Imputar (moda) | Solo 2 missings, y sí correla con el target (embarcar en "C" cambia la tasa de supervivencia) |
| `Fare` | Transformar (log1p) + escalar | Distribución con cola muy larga |
| `Sex`, `Embarked` | One-Hot Encoding | Pocas categorías, sin orden natural |
| `Pclass`, `SibSp`, `Parch` | Dejar tal cual | Ya son numéricas/ordinales, sin necesidad de tratamiento adicional |

*(Si quieres el razonamiento completo con los gráficos y tests, está en el notebook
de referencia — hoy nos centramos en el "cómo lo empaquetamos", no en el "qué decisión tomar").*


## Parte 2 — ¿Qué es un Pipeline?

### Transformer

Un **transformer** es cualquier objeto de sklearn que implementa `fit()` y `transform()`.
`fit()` aprende algo de los datos (una media, unas categorías...), `transform()` aplica
esa transformación aprendida.

- Ejemplos: `StandardScaler`, `SimpleImputer`, `OneHotEncoder`

### Pipeline

Un `Pipeline` es una lista de transformers que se ejecutan en orden. El último paso
puede ser un modelo (algo con `predict()`), no solo un transformer.

```python
mi_pipeline = Pipeline([
    ("paso_a", TransformerA()),
    ("paso_b", TransformerB()),
    ("modelo", LogisticRegression())
])
```

- `mi_pipeline.fit(X, y)` → equivale a encadenar `fit_transform` en cada paso, y `fit`
  en el último
- `mi_pipeline.predict(X)` → equivale a encadenar `transform` en cada paso, y `predict`
  en el último

**La idea clave**: una vez construido, el pipeline entero se comporta como un único
objeto de sklearn. Se guarda como un único objeto. Se carga como un único objeto.
Todo el preprocesado viaja pegado al modelo — no hay manera de aplicar el modelo sin
aplicar también el preprocesado, porque son la misma cosa.

### ColumnTransformer

`ColumnTransformer` es el complemento que nos permite aplicar transformers *distintos*
a columnas *distintas* del mismo dataframe — cada feature necesita su propio tratamiento.


## Parte 3 — Construyendo el pipeline

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

COLUMNS_TO_EXCLUDE = ["PassengerId", "Name", "Cabin", "Ticket", "Survived"]

Empezamos por las categóricas — `Sex` y `Embarked` necesitan imputación y encoding, en ese orden:

In [6]:
cat_pipeline = Pipeline([
    ("imputar_moda", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(drop="if_binary", handle_unknown="ignore")),
])

`drop="if_binary"` elimina una columna redundante cuando la variable solo tiene 2
categorías (evita la multicolinealidad de tener `Sex_male` y `Sex_female` cuando una
se deduce de la otra). `handle_unknown="ignore"` evita que el pipeline explote si en
producción llega una categoría que nunca vio en entrenamiento — importante, esto pasa
más de lo que crees.

Ahora `Fare`, que necesita imputación + log + escalado:


In [7]:
fare_pipeline = Pipeline([
    ("imputar_mediana", SimpleImputer(strategy="median")),
    ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("escalar", StandardScaler()),
])

Y `Age`, que solo necesita imputación + escalado:

In [8]:
age_pipeline = Pipeline([
    ("imputar_mediana", SimpleImputer(strategy="median")),
    ("escalar", StandardScaler()),
])

Ahora unimos todo con `ColumnTransformer`, indicando qué transformer va con qué columnas:

In [9]:
preprocessing = ColumnTransformer([
    ("fare", fare_pipeline, ["Fare"]),
    ("age", age_pipeline, ["Age"]),
    ("categoricas", cat_pipeline, ["Sex", "Embarked"]),
    ("excluir", "drop", COLUMNS_TO_EXCLUDE),
], remainder="passthrough")  # remainder="passthrough" -> el resto de columnas (Pclass, SibSp, Parch) se dejan tal cual

In [10]:
resultado = preprocessing.fit_transform(train)
df_resultado = pd.DataFrame(resultado, columns=preprocessing.get_feature_names_out())
df_resultado.head()

,fare__Fare,age__Age,categoricas__Sex_male,categoricas__Embarked_C,categoricas__Embarked_Q,categoricas__Embarked_S,remainder__Pclass,remainder__SibSp,remainder__Parch
0,-0.879741,-0.565736,1.0,0.0,0.0,1.0,3.0,1.0,0.0
1,1.361220,0.663861,0.0,1.0,0.0,0.0,1.0,1.0,0.0
2,-0.798540,-0.258337,0.0,0.0,0.0,1.0,3.0,0.0,0.0
3,1.062038,0.433312,0.0,0.0,0.0,1.0,1.0,1.0,0.0
4,-0.784179,0.433312,1.0,0.0,0.0,1.0,3.0,0.0,0.0


Por último, añadimos el modelo como último paso del pipeline:

In [11]:
from sklearn.ensemble import RandomForestClassifier

pipeline_completo = Pipeline([
    ("preprocesado", preprocessing),
    ("modelo", RandomForestClassifier(random_state=42)),
])

pipeline_completo

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocesado', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('fare', ...), ('age', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different t

Fíjate en algo importante: **le pasamos el `train` completo, con `Survived` y con
`Name` incluidos**. El pipeline se encarga de tirar lo que no necesita. Nosotros ya
no tenemos que acordarnos de excluir columnas cada vez que llamamos al modelo.


In [12]:
from sklearn.model_selection import cross_val_score

y_train = train["Survived"]
scores = cross_val_score(pipeline_completo, train, y_train, cv=5, scoring="balanced_accuracy")
print("Balanced accuracy (CV-5):", round(np.mean(scores), 4))

Balanced accuracy (CV-5): 0.7996


## Parte 4 — ¿Y si hacemos lo mismo con funciones sueltas?

Es una forma completamente válida de trabajar, y mucha gente la usa. Vamos a construir
el "mismo" preprocesado con funciones normales de Python, y comparamos.


In [13]:
def preprocess_data_funcion(X,
                            mean_col_imputed=["Age"],
                            mode_col_imputed=["Embarked"],
                            cols_to_encode=["Sex", "Embarked"],
                            cols_to_scale=["Age", "Fare"]):
    X_temp = X[[c for c in X.columns if c not in COLUMNS_TO_EXCLUDE]].copy()

    for col in mean_col_imputed:
        valor = round(X_temp[col].mean(), 0)
        X_temp.loc[X_temp[col].isna(), col] = valor

    for col in mode_col_imputed:
        valor = X_temp[col].mode().values[0]
        X_temp.loc[X_temp[col].isna(), col] = valor

    X_temp = pd.get_dummies(X_temp, columns=cols_to_encode, dtype=int)

    scaler = StandardScaler()
    X_temp[cols_to_scale] = scaler.fit_transform(X_temp[cols_to_scale])

    return X_temp

X_funcion = preprocess_data_funcion(train)
X_funcion.head()

,Pclass,Age,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
0,3,-0.597055,1,0,-0.502445,0,1,0,0,1
1,1,0.634162,1,0,0.786845,1,0,1,0,0
2,3,-0.289251,0,0,-0.488854,1,0,0,0,1
3,1,0.403309,1,0,0.420730,1,0,0,0,1
4,3,0.403309,0,0,-0.486337,0,1,0,0,1


In [14]:
cv_pipeline = cross_val_score(pipeline_completo, train, y_train, cv=5, scoring="balanced_accuracy")
cv_funcion = cross_val_score(RandomForestClassifier(random_state=42), X_funcion, y_train, cv=5, scoring="balanced_accuracy")

print("Pipeline:", round(np.mean(cv_pipeline), 4))
print("Función :", round(np.mean(cv_funcion), 4))

Pipeline: 0.7996
Función : 0.7963


> **¿Son exactamente el mismo preprocesado? Miradlo con atención antes**
> **de seguir leyendo.**

Fijaos en el parámetro de la función: `mean_col_imputed=["Age"]`. Imputa `Age` con la
**media**. El pipeline imputa `Age` con la **mediana**. Compruébalo:


In [15]:
print("Mediana de Age (lo que usa el pipeline):", train.Age.median())
print("Media de Age, redondeada (lo que usa la función):", round(train.Age.mean(), 0))

Mediana de Age (lo que usa el pipeline): 28.0
Media de Age, redondeada (lo que usa la función): 30.0


28 años frente a 30. No es un error que salte a la vista — el código corre sin fallos,
los números de CV son parecidos, y aun así los dos modelos **no están haciendo lo mismo**.

Esto no es un caso inventado para la clase. Es exactamente el tipo de bug que aparece
cuando la misma lógica de preprocesado se escribe dos veces en dos sitios distintos —
una vez la decides ("vamos a usar la mediana") y otra vez la implementas (por prisa,
por copiar de otro notebook, por lo que sea) de una forma sutilmente distinta. Con
`Pipeline`, el preprocesado se define **una sola vez**. No hay una segunda copia que
pueda divergir.


## Parte 5 — Ajustando hiperparámetros dentro de un Pipeline

Si queremos tunear el modelo con `GridSearchCV`, necesitamos decirle a qué paso del
pipeline pertenece cada hiperparámetro. La sintaxis es `nombre_del_paso__parametro`
(doble guion bajo):


In [16]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

pipeline_xgb = Pipeline([
    ("preprocesado", preprocessing),
    ("modelo", XGBClassifier(random_state=42, eval_metric="logloss")),
])

param_grid = {
    "modelo__n_estimators": [100, 200],
    "modelo__max_depth": [3, 5],
}

grid = GridSearchCV(pipeline_xgb, param_grid, cv=5, scoring="balanced_accuracy", n_jobs=-1)
grid.fit(train, y_train)

print("Mejor score (CV):", round(grid.best_score_, 4))
print("Mejores parámetros:", grid.best_params_)

Mejor score (CV): 0.8141
Mejores parámetros: {'modelo__max_depth': 3, 'modelo__n_estimators': 200}


`grid.best_estimator_` ya es el pipeline completo reentrenado con los mejores
parámetros sobre todos los datos de train — preprocesado incluido. No hace falta
reconstruir nada a mano.


## Parte 6 — Guardando el modelo

Vamos a guardar dos cosas para comparar en la siguiente parte: el pipeline ganador, y
el modelo entrenado sobre los datos de la función (sin el preprocesado embebido).


In [17]:
import joblib

mejor_pipeline = grid.best_estimator_
joblib.dump(mejor_pipeline, "modelo_pipeline.joblib")

modelo_funcion = XGBClassifier(random_state=42, eval_metric="logloss", n_estimators=100, max_depth=3)
modelo_funcion.fit(X_funcion, y_train)
joblib.dump(modelo_funcion, "modelo_funcion.joblib")

print("Modelos guardados.")

Modelos guardados.


## Parte 7 — Testeo

Ahora vamos a simular exactamente lo que pasará mañana: alguien (quizá tú mismo, quizá
el equipo de Ingeniería) abre un notebook o script **nuevo**, sin nada en memoria, y
solo tiene acceso a los archivos `.joblib` que acabas de guardar.

> ### REINICIA EL KERNEL AHORA (Kernel → Restart Kernel)
>
> No ejecutes las celdas de arriba otra vez. La idea es que en este punto no tengamos
> **nada** en memoria — ni `preprocessing`, ni `preprocess_data_funcion`, ni
> `COLUMNS_TO_EXCLUDE`. Solo lo que puedas cargar desde cero.

Cuando hayas reiniciado, ejecuta solo las celdas de aquí en adelante.


In [18]:
# Solo lo mínimo. Nada de lo definido arriba existe ya en memoria.
import joblib
import pandas as pd
from sklearn.metrics import classification_report

test = pd.read_csv("./data/titanic_test.csv")
X_test = test
y_test = test["Survived"]

### Intento 1 — el modelo-pipeline

In [19]:
modelo_pipeline = joblib.load("modelo_pipeline.joblib")
predicciones = modelo_pipeline.predict(X_test)

print(predicciones[:10])
print(classification_report(y_test, predicciones))

[0 0 0 0 0 0 0 0 1 0]
              precision    recall  f1-score   support

           0       0.78      0.82      0.80       260
           1       0.68      0.63      0.65       158

    accuracy                           0.75       418
   macro avg       0.73      0.72      0.73       418
weighted avg       0.74      0.75      0.74       418



Funciona. Sin que hayamos definido nada de preprocesado en este notebook nuevo. El
pipeline lleva todo consigo.

### Intento 2 — el modelo-función


In [20]:
modelo_funcion = joblib.load("modelo_funcion.joblib")
predicciones_funcion = modelo_funcion.predict(X_test)  # esto va a fallar

ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:Name: object, Sex: object, Ticket: object, Cabin: object, Embarked: object

Si has llegado hasta aquí ejecutando esta celda, te habrá saltado un error de XGBoost
quejándose de columnas de texto (`Name`, `Sex`, `Ticket`, `Cabin`, `Embarked`). El
modelo espera números — pero nadie le pasó los datos por `preprocess_data_funcion()`
antes de predecir, porque esa función **no viaja con el modelo guardado**. Vive en un
notebook que quien despliega esto ni siquiera tiene por qué haber visto.

> **¿Qué tendría que hacer ahora mismo alguien del equipo de Ingeniería**
> **para arreglar esto? ¿Y si la función tuviera un bug y nadie se diera cuenta durante**
> **semanas?**

Este es el motivo real por el que en cualquier empresa que despliega modelos de
sklearn en producción, veréis `Pipeline` — no es una preferencia estética de la
documentación de sklearn, es lo que evita que este tipo de fallo llegue a producción.


## Parte 8 — De notebook a repositorio real

Un notebook no se despliega. Nadie programa un notebook en un cron, ni lo pone detrás
de una API. En una empresa real, lo de hoy se organizaría así:

```
titanic-pipeline-demo/
├── data/
│   ├── raw/              # Datos de entrada, tal como llegan
│   └── processed/        # Datos intermedios (si hicieran falta)
├── models/               # Modelos entrenados (.joblib) — normalmente NO se versiona en git
├── notebooks/            # Exploración y prototipado — como el de hoy. Nunca producción
├── src/
│   ├── pipeline.py       # Única definición del preprocesado — la fuente de verdad
│   ├── train.py          # Entrena y guarda el modelo
│   └── predict.py        # Carga el modelo y predice
├── requirements.txt
└── .gitignore
```

**Importante**: `src/pipeline.py` se define **una sola vez**. `train.py` lo
importa para entrenar. `predict.py` ni siquiera lo necesita — solo carga el `.joblib`
y llama a `.predict()`. Así es exactamente como acabamos de ver que funciona el
modelo-pipeline: el código de predicción no sabe nada de preprocesado porque no lo
necesita saber.

Os dejo este mismo ejemplo montado como repo de verdad — con `train.py` y `predict.py`
ejecutándose desde terminal, no desde notebook. Podéis correr:

```bash
python src/train.py
python src/predict.py
```

y veréis los mismos resultados que hemos visto aquí, pero como se vería en producción.


## Cierre — ¿Pipeline siempre? No siempre

`Pipeline` no es la solución universal. Algunos casos donde cuesta más de lo que
aporta:

- **Feature engineering que cruza filas** (por ejemplo, una feature que depende de
  agregados por grupo, tipo "media de Fare por Pclass calculada sobre TODO el
  dataset") no encaja de forma natural en la API `fit/transform` por columna — se
  puede hacer con un `FunctionTransformer` a medida, pero se complica.
- **Debugging.** Cuando algo falla dentro de un pipeline muy anidado, encontrar qué
  paso exacto ha fallado es más difícil que con funciones sueltas y prints por el
  camino.
- **Prototipado muy rápido y desechable.** Si estás explorando un dataset por primera
  vez y no sabes ni qué decisiones vas a tomar, forzar todo a un pipeline desde el
  minuto uno puede ir más lento que trabajar con funciones sueltas y consolidar en
  pipeline cuando ya sabes qué quieres hacer.

La regla práctica: **explora con lo que te sea más rápido, pero antes de guardar un
modelo que otra persona (o tu yo del futuro) vaya a cargar sin ti al lado, empaquétalo
en un Pipeline.**

### Resumen

1. Un `Pipeline` encadena transformers y opcionalmente un modelo final
2. `ColumnTransformer` te permite aplicar transformers distintos a columnas distintas
3. El pipeline completo se guarda y se carga como **un único objeto** — preprocesado incluido
4. Duplicar la lógica de preprocesado en funciones sueltas es una fuente real de bugs silenciosos
5. `nombre_paso__parametro` es la sintaxis para tunear hiperparámetros dentro de un pipeline con GridSearchCV
6. En un repo real, el pipeline vive en un único módulo (`src/pipeline.py`) que se importa, no se copia
